# Modelamiento ATU — índice (un modelo por sistema)

**Decisión:** no un modelo Lima-global. Un notebook / fit por tipo de transporte.

| Sistema | Notebook | Outputs |
|---------|----------|---------|
| Metro L1 | [`Modelamiento_metro_l1.ipynb`](Modelamiento_metro_l1.ipynb) | `outputs/metro_l1/` |
| Troncal | [`Modelamiento_troncal.ipynb`](Modelamiento_troncal.ipynb) | `outputs/troncal/` |
| Corredores | [`Modelamiento_corredores.ipynb`](Modelamiento_corredores.ipynb) | `outputs/corredor/` |
| Alimentadores | [`Modelamiento_alimentadores.ipynb`](Modelamiento_alimentadores.ipynb) | `outputs/alimentador/` |

Helpers: [`model_atu_common.py`](model_atu_common.py).

Cada notebook: Chronos walk-forward + baselines + aforo proxy + regla U + anomalías.


## Comparativa rápida WMAPE (Chronos vs baselines)


In [3]:

from pathlib import Path
import pandas as pd

RAIZ = Path.cwd().resolve()
if not (RAIZ / "outputs").exists():
    RAIZ = RAIZ / "code" / "model" if (RAIZ / "code" / "model").exists() else RAIZ
# locate model dir
for cand in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (cand / "code" / "model" / "outputs").exists():
        MOD = cand / "code" / "model"
        break
    if (cand / "outputs" / "metro_l1").exists():
        MOD = cand
        break
else:
    raise FileNotFoundError("outputs no encontrados")

rows = []
for sis in ["metro_l1", "troncal", "corredor", "alimentador"]:
    p = MOD / "outputs" / sis / "metricas_modelos.csv"
    if not p.exists():
        continue
    m = pd.read_csv(p, index_col=0)
    for modelo, r in m.iterrows():
        rows.append({"sistema": sis, "modelo": modelo, **r.to_dict()})
comp = pd.DataFrame(rows)
print(comp.pivot_table(index="sistema", columns="modelo", values="WMAPE").round(3).to_string())
print()
print(comp.pivot_table(index="sistema", columns="modelo", values="MAE").round(1).to_string())


modelo       chronos  hora_media  naive_7d
sistema                                   
alimentador    0.507       0.624     0.658
corredor       0.311       0.353     0.410
metro_l1       0.155       0.198     0.193
troncal        0.310       0.329     0.412

modelo       chronos  hora_media  naive_7d
sistema                                   
alimentador     37.9        46.2      49.5
corredor       173.9       234.9     239.8
metro_l1       131.5       175.6     165.1
troncal         74.0        94.9     103.1


### Lectura

- **WMAPE** comparable entre sistemas; **MAE** no (escalas distintas).
- Chronos gana en los cuatro.
- Alimentadores: WMAPE más alto → más irregularidad / señal débil; modelo propio justificado.
- El notebook monolítico anterior (`Modelamiento_ATU.ipynb` mezclado L1+troncal) queda
  como referencia histórica; **usar los 4 notebooks por sistema**.
